In [2]:
import json
import string
from nltk.corpus import stopwords

In [3]:
# Loading the lexicon in the RAM
with open('../JSON Files/lexicon.json') as f:
    lexicon = json.load(f)
print('Lexicon Loaded')

Lexicon Loaded


In [4]:
def generate_ngrams(word, n=3):
    return [word[i:i+n] for i in range(len(word) - n + 1)]

In [5]:
def jaccard_similarity(grams1, grams2):
    intersection = set(grams1).intersection(set(grams2))
    union = set(grams1).union(set(grams2))
    return len(intersection) / len(union)

In [6]:
def get_closest_match(query, lexicon, n=3):
    query_ngrams = generate_ngrams(query, n)
    best_match = None
    best_score = 0

    for word in lexicon:
        word_ngrams = generate_ngrams(word, n)
        score = jaccard_similarity(query_ngrams, word_ngrams)

        if score > best_score:
            best_score = score
            best_match = word
    
    return best_match

In [7]:
def get_barrel(word_id):
    # For barrel_10_1 to barrel_10_1000 (First 10000 words)
    if word_id < 10000:
        barrel_index = (word_id // 10) + 1
        return f'../Barrels/barrel_10/barrel_10_{barrel_index}.json'
    
    # For barrel_250_1 to barrel_250_80 (Next 20000 words)
    elif word_id >= 10000 and word_id < 30000:
        barrel_index = ((word_id-10000) // 250) + 1
        return f'../Barrels/barrel_250/barrel_250_{barrel_index}.json'
    
    # For barrel_10000_1 to barrel_10000_46 (Remaining words)
    else:
        barrel_index = (word_id // 10000) - 2
        return f'../Barrels/barrel_10000/barrel_10000_{barrel_index}.json'

In [8]:
def calculate_score(posting):
    score = 0

    # Getting the term frequency from the title and the text of document
    tf_title = posting['positions'].get('title', 0)
    tf_text = posting['positions'].get('text', 0)

    # Calculating the score
    score = (tf_title*10) + (tf_text*0.5)  # Giving more weight to the title than the text
    return score

In [9]:
def rank_documents(postings_list):
    ranked_docs = []
    
    for doc_id, posting in postings_list.items():
        score = calculate_score(posting)
        ranked_docs.append({'ID': doc_id, 'score': score})

    # Sorting the documents based on the score in descending order
    ranked_docs.sort(key=lambda x: x['score'], reverse=True)

    return ranked_docs

In [10]:
def single_word_search(query, lexicon):
    query = query.lower()  # Normalizing the query

    if query not in lexicon:
        closest_match = get_closest_match(query, lexicon)
        print(f'No exact match found.\nInstead, showing results for {closest_match}')
        query = closest_match

    word_id = lexicon[query]
    print(f'The word id is {word_id}')
    if word_id is None:
        return f"Word '{query}' not found in the lexicon"
    
    barrel_file = get_barrel(word_id)
    if barrel_file is None:
        return f"Word '{query}' not found in the inverted index"
    
    try:
        with open(barrel_file, 'r') as f:
            barrel = json.load(f)
            print(f'{barrel_file} loaded.')

            if str(word_id) in barrel:
                return barrel[str(word_id)]['postings']
            else:
                return f"Word '{query}' not found in the barrel file"
        
    except FileNotFoundError:
        return f"Barrel file '{barrel_file}' not found"
    
    return None

In [11]:
def multiple_word_search(query_string, lexicon=lexicon):
    original_query = query_string
    print(f"Query: {original_query}")

    # Removing punctuation marks from the query
    for char in query_string:
        if char in string.punctuation:
            query_string = query_string.replace(char, '')  # Removing punctuation marks from the query

    query_words = query_string.lower().split()  # Normalizing and then splitting the query
    if not query_words:
        return f"No result found for query: '{original_query}'\nQuery contains only punctuation marks."
    
    query_words = [word for word in query_words if word not in stopwords.words('english')]  
    if not query_words:
        return f"Query '{original_query}' not found in the lexicon."
    
    print(f"Query words: {query_words}")

    # Getting the list of documents for each word in the query
    posting_list = []

    # Fetch posting list for each word in the query
    for word in query_words:
        postings = single_word_search(word, lexicon)

        if postings is not None and isinstance(postings, dict):
            posting_list.append(postings)  
    
    if not posting_list:
        return f"No result found for query: '{original_query}'"
    
    # Perform AND operation (intersection) on the posting lists
    intersection_docs = set(posting_list[0].keys())
    for postings in posting_list[1:]:
        intersection_docs.intersection_update(postings.keys())

    # Collecting final postings from intersection
    ranked_results = []
    if intersection_docs: 
        intersection_postings = {
            doc_id: postings[doc_id] for postings in posting_list for doc_id in intersection_docs
        }
        ranked_results = rank_documents(intersection_postings)

    # Perform OR operation (union) on the posting lists if the result of intersection is very small
    if len(intersection_docs) <= 3:
        union_docs = set()

        for posting in posting_list:
            union_docs.update(posting.keys())
        
        # Excluding intersection documents from the union
        union_docs.difference_update(intersection_docs)

        # Collecting final postings from union
        union_postings = {
            doc_id: postings[doc_id] for postings in posting_list for doc_id in union_docs if doc_id in postings
        }
        ranked_results.extend(rank_documents(union_postings))

    return ranked_results[:15]

In [12]:
query = 'the quick brown fox jumps over the lazy dog'
print(multiple_word_search(query, lexicon))

Query: the quick brown fox jumps over the lazy dog
Query words: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']
The word id is 2775
../Barrels/barrel_10/barrel_10_278.json loaded.
The word id is 3111
../Barrels/barrel_10/barrel_10_312.json loaded.
The word id is 8586
../Barrels/barrel_10/barrel_10_859.json loaded.
No exact match found.
Instead, showing results for jump
The word id is 4752
../Barrels/barrel_10/barrel_10_476.json loaded.
The word id is 3869
../Barrels/barrel_10/barrel_10_387.json loaded.
The word id is 3561
../Barrels/barrel_10/barrel_10_357.json loaded.
[{'ID': 'doc165234', 'score': 3.0}, {'ID': 'doc95571', 'score': 1.5}, {'ID': 'doc42074', 'score': 1.0}, {'ID': 'doc14040', 'score': 1.0}, {'ID': 'doc128279', 'score': 1.0}, {'ID': 'doc19542', 'score': 1.0}, {'ID': 'doc107273', 'score': 0.5}, {'ID': 'doc156388', 'score': 0.5}]


<hr>